# TASK 1: DATA COLLECTION

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("TASK 1: THU THẬP VÀ KIỂM TRA DỮ LIỆU")

TASK 1: THU THẬP VÀ KIỂM TRA DỮ LIỆU


## 1.0. CONVERT .DAT (LATIN-1) TO .CSV (UTF-8)

In [2]:
print("\n[0/4] Converting .dat (latin-1) → .csv (utf-8)...")
# Check if CSV files already exist
csv_exists = (
    os.path.exists('../data/raw/movies.csv') and
    os.path.exists('../data/raw/ratings.csv') and
    os.path.exists('../data/raw/users.csv')
)

if not csv_exists:
    print("Converting .dat files to .csv...")
    
    # Read movies.dat (trực tiếp trong data/raw/)
    movies_dat = pd.read_csv(
        '../data/raw/movies.dat',
        sep='::',
        engine='python',
        names=['movieId', 'title', 'genres'],
        encoding='latin-1'
    )
    movies_dat.to_csv('../data/raw/movies.csv', index=False, encoding='utf-8')
    print(f"movies.csv:  {len(movies_dat):,} rows")
    
    # Read ratings.dat (trực tiếp trong data/raw/)
    ratings_dat = pd.read_csv(
        '../data/raw/ratings.dat',
        sep='::',
        engine='python',
        names=['userId', 'movieId', 'rating', 'timestamp'],
        encoding='latin-1'
    )
    ratings_dat.to_csv('../data/raw/ratings.csv', index=False, encoding='utf-8')
    print(f"ratings.csv: {len(ratings_dat):,} rows")
    
    # Read users.dat (trực tiếp trong data/raw/)
    users_dat = pd.read_csv(
        '../data/raw/users.dat',
        sep='::',
        engine='python',
        names=['userId', 'gender', 'age', 'occupation', 'zip_code'],
        encoding='latin-1'
    )
    users_dat.to_csv('../data/raw/users.csv', index=False, encoding='utf-8')
    print(f"users.csv:   {len(users_dat):,} rows")
    
    print("Conversion completed: latin-1 → utf-8")
else:
    print("CSV files đã tồn tại, skip conversion!")


[0/4] Converting .dat (latin-1) → .csv (utf-8)...
CSV files đã tồn tại, skip conversion!


## 1.1. LOAD DATASET

In [3]:
print("\n[1/4] Loading MovieLens 1M dataset...")
movies = pd.read_csv('../data/raw/movies.csv')
ratings = pd.read_csv('../data/raw/ratings.csv')
users = pd.read_csv('../data/raw/users.csv')
print("Load thành công!")


[1/4] Loading MovieLens 1M dataset...
Load thành công!


## 1.2. KIỂM TRA YÊU CẦU ĐỀ BÀI

In [4]:
print("\n[2/4] Kiểm tra yêu cầu đề bài...")

print(f"\nYêu cầu 1: Số items ≥ 2,000")
print(f"→ Số movies: {len(movies):,}")

print(f"\nYêu cầu 2: Có ≥ 5 features")
print(f"→ Features hiện tại: {movies.columns.tolist()}")
print(f"→ Số features: {len(movies.columns)}")


[2/4] Kiểm tra yêu cầu đề bài...

Yêu cầu 1: Số items ≥ 2,000
→ Số movies: 3,883

Yêu cầu 2: Có ≥ 5 features
→ Features hiện tại: ['movieId', 'title', 'genres']
→ Số features: 3


# 1.3. THỐNG KÊ DATASET

In [5]:
print("\n[3/4] Thống kê tổng quan...")

print(f"\nThống kê cơ bản:")
print(f"Tổng số movies:  {len(movies):,}")
print(f"Tổng số ratings: {len(ratings):,}")
print(f"Tổng số users:   {ratings['userId'].nunique():,}")
print(f"Tổng số users (verified): {len(users):,}")

print(f"\nRatings distribution:")
print(ratings['rating'].value_counts().sort_index())

print(f"\nRating statistics:")
print(ratings['rating'].describe())


[3/4] Thống kê tổng quan...

Thống kê cơ bản:
Tổng số movies:  3,883
Tổng số ratings: 1,000,209
Tổng số users:   6,040
Tổng số users (verified): 6,040

Ratings distribution:
rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

Rating statistics:
count    1.000209e+06
mean     3.581564e+00
std      1.117102e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


## 1.4. TẠO THÊM FEATURES (OPTIMIZED VERSION)

In [6]:
print("\n[4/4] Tạo thêm features để đủ yêu cầu...")

# Feature 1,2,3: movieId, title, genres (có sẵn)
print("Feature 1: movieId")
print("Feature 2: title")  
print("Feature 3: genres")

# Feature 4: Extract year từ title
if 'year' not in movies.columns:
    movies['year'] = movies['title'].str.extract(r'\((\d{4})\)', expand=False)
    movies['year'] = pd.to_numeric(movies['year'], errors='coerce')
    print("Feature 4: year")

# Feature 5,6,7: Tính rating stats
if 'rating_avg' not in movies.columns:
    rating_stats = ratings.groupby('movieId').agg({
        'rating': ['mean', 'count', 'std']
    }).reset_index()
    rating_stats.columns = ['movieId', 'rating_avg', 'rating_count', 'rating_std']
    
    movies = movies.merge(rating_stats, on='movieId', how='left')
    print("Feature 5: rating_avg")
    print("Feature 6: rating_count")  
    print("Feature 7: rating_std")

# Feature 8: Clean title (MovieLens 1M không có tags)
if 'title_clean' not in movies.columns:
    movies['title_clean'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()
    print("Feature 8: title_clean")

# Feature 9: Genre count (thay thế tags)
if 'genre_count' not in movies.columns:
    movies['genre_count'] = movies['genres'].str.count('\|') + 1
    print("Feature 9: genre_count")

# Feature 10: Popularity category (bonus)
if 'popularity_category' not in movies.columns:
    movies['popularity_category'] = pd.cut(
        movies['rating_count'],
        bins=[0, 10, 50, 200, float('inf')],
        labels=['Niche', 'Moderate', 'Popular', 'Blockbuster']
    )
    print("Feature 10: popularity_category")
    
# Feature 11: Decade (thập kỷ)
if 'decade' not in movies.columns:
    movies['decade'] = (movies['year'] // 10) * 10
    print("Feature 11: decade")

# XÓA CÁC CỘT TRÙNG LẶP (nếu có)
duplicate_cols = [col for col in movies.columns if col.endswith('_x') or col.endswith('_y')]
if duplicate_cols:
    print(f"\nPhát hiện {len(duplicate_cols)} cột trùng lặp: {duplicate_cols}")
    
    # Xóa cột _x, giữ _y và rename
    cols_to_drop = [col for col in duplicate_cols if col.endswith('_x')]
    movies = movies.drop(columns=cols_to_drop)
    
    # Rename _y về tên gốc
    rename_dict = {col: col.replace('_y', '') for col in movies.columns if col.endswith('_y')}
    movies = movies.rename(columns=rename_dict)
    
    print("Đã xóa cột trùng lặp!")

print(f"\nTổng số features cuối cùng: {len(movies.columns)}")
print(f"Danh sách features: {movies.columns.tolist()}")



[4/4] Tạo thêm features để đủ yêu cầu...
Feature 1: movieId
Feature 2: title
Feature 3: genres
Feature 4: year
Feature 5: rating_avg
Feature 6: rating_count
Feature 7: rating_std
Feature 8: title_clean
Feature 9: genre_count
Feature 10: popularity_category
Feature 11: decade

Tổng số features cuối cùng: 11
Danh sách features: ['movieId', 'title', 'genres', 'year', 'rating_avg', 'rating_count', 'rating_std', 'title_clean', 'genre_count', 'popularity_category', 'decade']


## HIỂN THỊ KẾT QUẢ

In [7]:
print("TỔNG KẾT")

print(f"\nTổng số features hiện tại: {len(movies.columns)}")
print(f"Features: {movies.columns.tolist()}")

print(f"\nSample data với features mới:")
print(movies[['movieId', 'title_clean', 'year', 'genres', 'rating_avg', 'rating_count']].head(10))

print(f"\nMovies có nhiều ratings nhất:")
print(movies.nlargest(10, 'rating_count')[['title_clean', 'rating_avg', 'rating_count', 'genres']])

print(f"\nMovies có rating cao nhất (≥100 ratings):")
popular_movies = movies[movies['rating_count'] >= 100]
print(popular_movies.nlargest(10, 'rating_avg')[['title_clean', 'rating_avg', 'rating_count', 'genres']])

TỔNG KẾT

Tổng số features hiện tại: 11
Features: ['movieId', 'title', 'genres', 'year', 'rating_avg', 'rating_count', 'rating_std', 'title_clean', 'genre_count', 'popularity_category', 'decade']

Sample data với features mới:
   movieId                  title_clean  year                        genres  \
0        1                    Toy Story  1995   Animation|Children's|Comedy   
1        2                      Jumanji  1995  Adventure|Children's|Fantasy   
2        3             Grumpier Old Men  1995                Comedy|Romance   
3        4            Waiting to Exhale  1995                  Comedy|Drama   
4        5  Father of the Bride Part II  1995                        Comedy   
5        6                         Heat  1995         Action|Crime|Thriller   
6        7                      Sabrina  1995                Comedy|Romance   
7        8                 Tom and Huck  1995          Adventure|Children's   
8        9                 Sudden Death  1995                 

In [8]:
## LƯU FILE

In [9]:
print("LƯU FILE")

movies.to_csv('../data/raw/movies_with_features.csv', index=False, encoding='utf-8')
print("Đã lưu: data/raw/movies_with_features.csv")

# Lưu users 
users.to_csv('../data/raw/users_with_features.csv', index=False, encoding='utf-8')
print("Đã lưu: data/raw/users_with_features.csv")

# Lưu thống kê
stats = {
    'total_movies': len(movies),
    'total_ratings': len(ratings),
    'total_users': ratings['userId'].nunique(),
    'avg_rating': ratings['rating'].mean(),
    'total_features': len(movies.columns)
}
stats_df = pd.DataFrame([stats])
stats_df.to_csv('../data/raw/dataset_stats.csv', index=False)
print("Đã lưu: data/raw/dataset_stats.csv")

print("TASK 1 HOÀN THÀNH!")


LƯU FILE
Đã lưu: data/raw/movies_with_features.csv
Đã lưu: data/raw/users_with_features.csv
Đã lưu: data/raw/dataset_stats.csv
TASK 1 HOÀN THÀNH!
